## Monte Carlo Within Metropolis for Diffusion Models

### Motivation

This notebook is a continuation of `monte_carlo_within_metropolis.ipynb`, where we defined the Monte Carlo Within Metropolis (MCWM) algorithm for sampling from a posterior when likelihood evaluations require Monte Carlo estimation.

Where this notebook differs is the application of MCWM to diffusion processes. Here, we observe a process at discrete times, but its path between observations is hidden.

The latent Gaussian model (LGM) we used in `monte_carlo_within_metropolis.ipynb` contains a latent variable, leaving us with the likelihood
$$
p(y\mid\theta)=\int p(y,z\mid\theta)\,dz.
$$
We estimated this likelihood by averaging over simulated values of the latent variable. In that example, both the exact likelihood and posterior were available, so we used them to test the algorithm against a known benchmark.

Inference for diffusion models is similar in nature: we treat the unobserved path between two observed states as latent data. Let $y_i$ and $y_{i+1}$ be observations at consecutive observation times, and let $\theta$ denote the unknown diffusion parameters.

To approximate the transition density, we divide the observation interval into $M$ substeps and introduce intermediate states $z_1,\ldots,z_{M-1}$. Under this discretized model, the transition density is
$$
p^{(M)}(y_{i+1}\mid y_i,\theta)
=
\int
p^{(M)}(y_{i+1},z_1,\ldots,z_{M-1}\mid y_i,\theta)
\,dz_1\cdots dz_{M-1}.
$$
Here, $p^{(M)}$ denotes densities under the discretized model. The observed endpoints remain fixed while we integrate over the intermediate states.

MCWM can then use Monte Carlo estimates of these transition densities to perform approximate posterior sampling. This introduces an additional consideration: besides the likelihood-estimation sample size $N$, we must choose the path resolution $M$.

### In This Notebook

In this notebook, we first cover transitioning the MCWM theory and algorithmic information
presented in ```monte_carlo_within_metropolis.ipynb``` to the application of 
discretely-observed diffusion  processes. We finish by implement the theory and
algorithm to an example using the Ornstein–Uhlenbeck process.

### Theory

#### Diffusion Processes

A diffusion process is a solution to the stochastic differential equation (SDE):
$$
dX_t = \mu(X_t, \theta)dt + \sigma(X_t, \theta)\,dW_t, \quad 0 \leq t \leq T,
$$
where $X_t \in \mathbb{R}^d$, $\mu \in \mathbb{R}^{d}$ and $\nu = \sigma \sigma^{\top} 
\in \mathbb{R}^{d \times d}$ are drift and covariance coefficients respectively,
$\theta$ is the parameter vector, and $W_t$ is a $d$-dimensional Brownian motion (Wiener process).

##### Example: Ornstein-Uhlenbeck
The Ornstein-Uhlenbeck (OU) process is a mean-reverting stochastic process. Such a 
process tends to drift towards its long-run mean. In other words, one can see the OU
process as a random walk in continuous time but with a tendency for the walk to move back 
towards a central location, with a greater attraction when the process is further away
from the centre. 

OU is defined by the following SDE: 
$$
    dX_t = \kappa(m-X_t)dt + \sigma\,dW_t, \qquad \kappa>0, \quad \sigma>0,
$$
where $\kappa$ is the mean reversion speed/strength, $m$ is the long-run mean, and
$\sigma$ is the diffusion coefficient. The parameter vector $\theta$ contains whichever
of these parameters we wish to estimate; the remaining parameters are treated as known.

The OU process has a tractable Gaussian transition density. We will deliberately
approximate this density so that the exact likelihood provides a benchmark for our method.

#### Diffusions in Bayesian Inference and the Need for MCWM

In statistics, one may use a diffusion process as a model to describe, say, temporal-based data.
In a Bayesian setting, it becomes of interest to perform inference on the parameters $\theta$ that
describes $\mu$ and $\sigma$. The primary mechanism to do so is via the posterior density $p(\theta \mid y)$.
Unlocking the posterior allows one to quantify uncertainty about 
the estimated parameters after observing evidence. The posterior is proportional the 
likelihood of the data given the parameters multiplied by the prior of the parameters:
$$
    p(\theta \mid y) \propto p(y \mid \theta)p(\theta).
$$

When attempting to estimate the parameters $\theta$ in a continuous-time Markov process,
the reality of the world only allows us to capture observations at discrete time points
(usually taken at equally spaced time intervals). By the Markov property, the joint likelihood
of these observations can be written as a product of the transition densities:
$$
    p(y \mid \theta) = \prod_{i=0}^{n-1} p(\Delta_i, y_{i+1} \mid y_i, \theta),
$$
where $\Delta_i=t_{i+1}-t_i$ is the time between observations, $y_i$ is the $i$-th observed
state, $y_{i+1}$ is the $i+1$ observed state $(i=0,\ldots,n-1)$, and $\theta$ contains the diffusion parameters.

The likelihood $p(\Delta_i, \cdot \mid \cdot, \theta)$ is usually intractable for 
this class of models. This motivates the idea of approximating the transition densities.

#### Approximating the Transition Densities

Since for many diffusions the transition density is intractable, we need a way 
When the interval between two observed states is sufficiently small, one can use
the Euler approximation to the transition density. The Euler approximation to the 
transition density is a manageable Gaussian transition.

In order to shrink said interval,
we divide each observation interval into $M$ substeps of length
$$
h_i=\frac{\Delta_i}{M}.
$$
For interval $i$, we write
$$
u_{i,0}=y_i,
\qquad
u_{i,M}=y_{i+1},
\qquad
u_i=(u_{i,1},\ldots,u_{i,M-1}).
$$
The endpoints are observed and fixed. The $M-1$ intermediate states form the
unobserved discretized path. Under this notation, the Euler approximation to the
transition density is:
$$
\begin{align}
    p^{(M)}(\Delta_i, y_{i+1} \mid y_i, \theta) = 
    \int \prod_{m=0}^{M-1}\phi(u_{m+1};\, u_m + h_i\mu(u_m, \theta), h\nu(u_m, \theta))du_1,\ldots, du_{M-1},
\end{align}
$$
where, $\phi(\cdot;\, \mu, \nu)$ denotes the Gaussian density with mean $\mu$ and
covariance $\nu$. While the overall integral is still difficult to compute, the factors
inside the integral are tractable. This allows us to estimate the posterior distribution
as:
$$
    \pi^{(M)}(\theta \mid y) \propto p(\theta)\, \prod_{i=0}^{n-1}p^{(M)}(\Delta_i, y_{i+1} \mid y_i, \theta).
$$

Note: we'll assume equally spaced intervals between observations so that we can drop the
$i$ subscript on $\Delta$ and $h$ going forward.

#### Estimating the Transition Density Via Importance Sampling

The integral in $(1)$ can be evaluated using importance sampling. Writing the
transition density as an expectation with respect to an importance sampling 
density, $q(\cdot)$ defined on $\mathbb{R}^{d \times M-1}$, we have: 
$$
    p^{(M)}(\Delta, y_{i+1} \mid y_i, \theta) = \mathbb{E}_q[R_M(U)], \quad R_M(U) = \frac{\prod_{m=0}^{M-1}\phi(u_{m+1};\, u_m + h\mu(u_m, \theta), h\nu(u_m, \theta))}{q(U)}, \quad U = (U_1, \ldots, U_{M-1}),
$$
where $\mathbb{E}_q$ is the expectation with respect to density $q$. The above expectation
cannot be evaluated however we may estimate it via a Monte Carlo integration weighted by
$R_M(U)$. This is done by first drawing $N$ indpendent paths using the importance sampling
density $q(\cdot)$, then computing the sample mean of $R_M(U)$. We'll denote this estimator
by:
$$
    p^{(M,N)}(\Delta, y_{i+1} \mid y_i, \theta) = \frac1N \sum_{k=1}^N R_M(U^{(k)}), \qquad (2)
$$
where $U^{(k)} = (U_{k,1}, \ldots, U_{k,M-1})$ is a random sample from $q$, $k=1,\ldots, N$.
This allows us to stochastically approximate the posterior as:
$$
    \pi^{(M,N)}(\theta \mid y) \propto p(\theta)\, \prod_{i=0}^{n-1} p^{(M,N)}(\Delta, y_{i+1} \mid y_i, \theta),
$$
where $p^{(M,N)}$ is defined as in $(2)$.

#### The Modified Brownian Bridge Proposal

Fix an interval $i$ and set 
$$
    U_{i,0} = y_i, \qquad U_{i,M} = y_{i+1}.
$$
In order to proceed, we need to choose an importance-sampler/proposal density $q$. A commonly
used proposal for a missing path between two observations $U^{(i)}$ is one induced by
the Modified Brownian Bridge (MBB) sampler. MBB leverages the information provided by
the known end points to generate plausible intermediate states. That information is 
used to guide each simualted path towards the endpoint, while reducing its conditional 
variance as the endpoint approaches. 

The MBB generates intermediate states as follows: for $j=0,\ldots, M-2$, intermediate states
are sequentially generated by:
$$
    U^{i, j+1} \mid U_{i,j} = u_{i,j} \sim N_d(a_{i,j}, C_{i,j}),
$$
where
$$
    a_{i,j} = u_{i,j} + \frac{y_{i+1} - u_{i,j}}{M - j}, \qquad C_{i,j} = h\,\frac{M - j - 1}{M - j} \nu(u_{i,j}, \theta).
$$

For generated path $u^{(i)} = (u_{i,1}, \ldots, u_{i, M-1})$, its proposal density is
$$
    q_i(u^{(i)} \mid y_{i}, y_{i+1}, \theta) = \prod_{j=0}^{M-2} \phi(u_{i, j+1};\, a_{i,j}, C_{i,j}).
$$

### The Algorithm: MCWM

We now combine the importance-sampling estimator with the Metropolis Hastings acceptance rule. We assume that the process is observed without measurement error and condition on the initial observation \(y_0\).

Choose the number of Euler substeps $M$, the number of simulated paths $N$, and an initial parameter value $\theta^{(0)}$. Keep $M$ and $N$ fixed throughout the run.

Starting at the current parameter value $\theta^{(r-1)}$, iteration $r$ proceeds as follows.

1. **Estimate the transition densities at the current parameter value.**

   For each observation interval $i=0,\ldots,n-1$, independently generate $N$ intermediate paths using the MBB proposal:
   $$
   U_{\mathrm{curr}}^{(i,k)}
   \sim q_i(\cdot\mid y_i,y_{i+1},\theta^{(r-1)}),
   \qquad k=1,\ldots,N.
   $$
   The endpoints of each path are fixed at $y_i$ and $y_{i+1}$. Evaluate the importance weights and calculate
   $$
   p_{i,\mathrm{curr}}^{(M,N)}
   =
   \frac{1}{N}
   \sum_{k=1}^{N}
   R_M\!\left(U_{\mathrm{curr}}^{(i,k)};\theta^{(r-1)}\right).
   $$
   Here, $$R_M(U;\theta)$$ denotes the importance weight defined previously, with its dependence on the parameter value made explicit. The interval is determined by $i$.

2. **Propose a new parameter value and estimate its transition densities.**

   Draw
   $$
   \theta^*\sim J(\cdot\mid\theta^{(r-1)}).
   $$
   Repeat the path simulation and importance-weight calculation at $\theta^*$:
   $$
   U_{\mathrm{prop}}^{(i,k)}
   \sim q_i(\cdot\mid y_i,y_{i+1},\theta^*),
   $$
   $$
   p_{i,\mathrm{prop}}^{(M,N)}
   =
   \frac{1}{N}
   \sum_{k=1}^{N}
   R_M\!\left(U_{\mathrm{prop}}^{(i,k)};\theta^*\right).
   $$
   These paths are generated independently of the current-state paths. All path simulations in this iteration are fresh, rather than retained from previous iterations.

3. **Calculate the noisy acceptance ratio.**

   Using the estimated transition densities, calculate
   $$
   \widehat r
   =
   \frac{
   p(\theta^*)
   \displaystyle\prod_{i=0}^{n-1}p_{i,\mathrm{prop}}^{(M,N)}
   }{
   p(\theta^{(r-1)})
   \displaystyle\prod_{i=0}^{n-1}p_{i,\mathrm{curr}}^{(M,N)}
   }
   \frac{
   J(\theta^{(r-1)}\mid\theta^*)
   }{
   J(\theta^*\mid\theta^{(r-1)})
   }.
   $$
   The acceptance probability is
   $$
   \alpha=\min(1,\widehat r).
   $$
   For a symmetric parameter proposal, the $J$ terms cancel.

4. **Accept or reject the proposed parameter value.**

   Draw $V_r\sim\operatorname{Uniform}(0,1)$, independently of the other simulations, and set
   $$
   \theta^{(r)}
   =
   \begin{cases}
   \theta^*, & V_r<\alpha,\\
   \theta^{(r-1)}, & \text{otherwise}.
   \end{cases}
   $$
   Store the resulting parameter value, including repeated values after rejection.

For numerical stability, we calculate the ratio on the log scale:
$$
\begin{aligned}
\log\widehat r
&=
\log p(\theta^*)-\log p(\theta^{(r-1)})\\
&\quad+
\sum_{i=0}^{n-1}
\left[
\log p_{i,\mathrm{prop}}^{(M,N)}
-
\log p_{i,\mathrm{curr}}^{(M,N)}
\right]\\
&\quad+
\log J(\theta^{(r-1)}\mid\theta^*)
-
\log J(\theta^*\mid\theta^{(r-1)}).
\end{aligned}
$$
The decision is then to accept when
$$
\log V_r<\min(0,\log\widehat r).
$$

The defining feature of MCWM is that both likelihood estimates are regenerated at every iteration, including after a rejection. We retain the current parameter when a proposal is rejected, but discard the simulated paths and likelihood estimates.

At finite $N$, this noisy acceptance mechanism generally does not preserve the Euler posterior $\pi^{(M)}(\theta\mid y)$ exactly. Increasing the chain length addresses sampling error, while $M$ and $N$ control the discretization and likelihood-estimation approximations.

This follows Algorithm 1 in [Stramer and Bognar](https://stat.uiowa.edu/sites/stat.uiowa.edu/files/2024-04/Bayesian-Inference-Irreducible-Diffusion-Processes-Using-Pseudo-Marginal-Approach.pdf).

### Example: Ornstein–Uhlenbeck Process